# Homework 1

#### EE-556 Mathematics of Data - Fall 2024


In this homework, we consider a multiclass classification task modeled by multinomial (softmax) logistic regression. Your goal will be to analyze the estimator and its properties (convexity, existence/uniqueness), and to derive gradients/Hessians and smoothness bounds. The first part consists of theoretical questions only.


<div class="alert alert-info">
  ℹ️ <strong>Information on group based work:</strong>
</div>

- You are to deliver only 1 notebook per group.
- Asking assistance beyond your group is ok, but answers should be individual to the group.
- In the event that there was <span style="color: red;">disproportional work done</span> by different group members, let the TAs know.
- Only one member of the group is allowed to use AI. We will require sharing the conversation history with the AI in the form of a public link. If you use multiple conversations across the same or multiple tools please share all of them. Name the person in your group who is allowed to use AI. We encourage you to use the AI to help you understand the material, but we ask you to write the code and theory solutions by yourself.

<div style="border: 1px solid #f00; background-color: #fdd; padding: 10px; border-radius: 5px;">
  ⚠️ Do not forget: Write who are the people in your group as well as their respective SCIPER numbers
</div>


Person 1 **Sofija Orlovic**: || Person 1 **SCIPER**: 408072


Person 2 **Petar Damjanovic**: || Person 2 **SCIPER**: 405185


Person 3 **Marko Stojanovic**: || Person 3 **SCIPER**: 408057


<div style="border: 1px solid #0a0; background-color: #dfd; padding: 10px; border-radius: 5px;">
  📓 Feedback on AI use: Please use the following cell to provide feedback on the AI use in this notebook.
  
  For example, how useful were the tools to you? Which tools did you use? Did you feel like they helped you understand the material better?
</div

## 1. Multiclass Softmax Logistic Regression - 15 Points


We now model multiclass classification with classes $c \in \{1,\dots,C\}$. For each sample $(\mathbf{a}_i, b_i)$ with $\mathbf{a}_i \in \mathbb{R}^p$ and $b_i \in \{1,\dots,C\}$, let $\mathbf{X} = [\mathbf{x}_1,\dots,\mathbf{x}_C] \in \mathbb{R}^{p\times C}$ be the class weight matrix. The softmax model defines

$$
\mathbb{P}(b_i = c \mid \mathbf{a}_i) = \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)}.
$$

Assume i.i.d. samples $\{(\mathbf{a}_i,b_i)\}_{i=1}^n$. Our goal is to estimate $\mathbf{X}$ by maximum likelihood (and later with an $\ell_2$ regularizer).


__(a)__ (1 point) Show that the negative log-likelihood $f$ can be written as:

$$
\begin{aligned}
 f(\mathbf{X})
 &= - \log \mathbb{P}(b_1,\dots,b_n\mid \mathbf{a}_1,\dots,\mathbf{a}_n)\\
 &= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right].
\end{aligned}
$$



**Answer.**  
We are given i.i.d. samples $\{(\mathbf{a}_i, b_i)\}_{i=1}^n$.  
By independence, the joint probability is

$$
\begin{aligned}
\mathbb{P}(b_1, \dots, b_n \mid \mathbf{a}_1, \dots, \mathbf{a}_n) 
&= \prod_{i=1}^n \mathbb{P}(b_i \mid \mathbf{a}_i).
\end{aligned}
$$

After taking the negative log-likelihood we get

$$
\begin{aligned}
f(\mathbf{X})
&= - \log \mathbb{P}(b_1, \dots, b_n \mid \mathbf{a}_1, \dots, \mathbf{a}_n) \\
&= - \sum_{i=1}^n \log \mathbb{P}(b_i \mid \mathbf{a}_i).
\end{aligned}
$$

From the softmax model we have

$$
\begin{aligned}
\mathbb{P}(b_i = c \mid \mathbf{a}_i) 
&= \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)}.
\end{aligned}
$$

Thus, for the observed label $b_i$,

$$
\begin{aligned}
- \log \mathbb{P}(b_i \mid \mathbf{a}_i) 
&= - \mathbf{a}_i^\top \mathbf{x}_{b_i} 
  + \log \left( \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right).
\end{aligned}
$$

Finally, summing over all samples yields

$$
\begin{aligned}
f(\mathbf{X}) 
&= \sum_{i=1}^n \left[ - \mathbf{a}_i^\top \mathbf{x}_{b_i} 
  + \log \left( \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right) \right]. \blacktriangle
\end{aligned}  
$$ 

__(b)__ (2 points) Show that $\mathbf{u} \mapsto \log\!\left(\sum_{k=1}^C e^{u_k}\right)$ is convex on $\mathbb{R}^C$. Then, show that $f(\mathbf{X})$ is convex.


Hint: use Jensen's inequality.

**Answer.**  

**First part. Convexity of $\phi(\mathbf{u})$.**

Declare the function whose convexity we want to prove:  $\phi(\mathbf{u})=\log\!\left(\sum_{k=1}^C e^{u_k}\right)$ for $\mathbf{u}\in\mathbb{R}^C$.
Now, by fix $\theta\in[0,1]$, taking $\mathbf{u},\mathbf{v}\in\mathbb{R}^C$ and setting $\mathbf{w}=\theta\mathbf{u}+(1-\theta)\mathbf{v}$
we have
$$
\phi(\mathbf{w})
=\log\!\left(\sum_{k=1}^C e^{w_k}\right)
=\log\!\left(\sum_{k=1}^C e^{\theta u_k+(1-\theta)v_k}\right)
=\log\!\left(\sum_{k=1}^C (e^{u_k})^{\theta}(e^{v_k})^{1-\theta}\right).
$$

Let's step back for a second and recall Hölder’s inequality: 
For vectors $x,y\in\mathbb{R}^C$ and $p\in[1,+\infty)$ with $\dfrac{1}{q}+\dfrac{1}{p}=1$,
  $$
  |x^{\top}y|\;\le\;\|x\|_{q}\,\|y\|_{p}.
  $$

We apply Hölder to the nonnegative sequences
$$
x_k=(e^{u_k})^{\theta},\qquad y_k=(e^{v_k})^{1-\theta},
$$
with
$$
p=\frac{1}{\,1-\theta\,},\qquad q=\frac{1}{\,\theta\,}.
$$
These choices require $\theta\in(0,1)$ so that $p,q\ge1$ and $\tfrac1q+\tfrac1p=1$.

Now, we compute the norms taking into account the substitutions we made:
$$
\|x\|_{q}
=\left(\sum_{k=1}^C x_k^{\,q}\right)^{\!1/q}
=\left(\sum_{k=1}^C (e^{u_k})^{\theta q}\right)^{\!1/q}
=\left(\sum_{k=1}^C e^{u_k}\right)^{\!1/q}
=\left(\sum_{k=1}^C e^{u_k}\right)^{\theta},
$$
since $\theta q=1$. Similarly,
$$
\|y\|_{p}
=\left(\sum_{k=1}^C y_k^{\,p}\right)^{\!1/p}
=\left(\sum_{k=1}^C (e^{v_k})^{(1-\theta)p}\right)^{\!1/p}
=\left(\sum_{k=1}^C e^{v_k}\right)^{\!1/p}
=\left(\sum_{k=1}^C e^{v_k}\right)^{1-\theta},
$$
since $(1-\theta)p=1$.

Hölder therefore gives (no absolute values needed because all terms are nonnegative)
$$
\sum_{k=1}^C (e^{u_k})^{\theta}(e^{v_k})^{1-\theta}
\;\le\;
\left(\sum_{k=1}^C e^{u_k}\right)^{\theta}
\left(\sum_{k=1}^C e^{v_k}\right)^{1-\theta}.
$$

Because $\log$ is increasing,

$$
\phi(\theta\mathbf{u}+(1-\theta)\mathbf{v})
=\log\!\left(\sum_{k=1}^C (e^{u_k})^{\theta}(e^{v_k})^{1-\theta}\right)
\le
\theta\log\!\left(\sum_{k=1}^C e^{u_k}\right)
+(1-\theta)\log\!\left(\sum_{k=1}^C e^{v_k}\right)
=
\theta\phi(\mathbf{u})+(1-\theta)\phi(\mathbf{v}).
$$

For $\theta=0$ or $\theta=1$, the inequality holds with equality.  
Combining these cases, $\phi$ is convex on $\mathbb{R}^C$.



**Step 2. Convexity of $f(\mathbf{X})$.**

Recall
$$
f(\mathbf{X}) 
= \sum_{i=1}^n \Big[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} 
+ \log \left( \sum_{k=1}^C e^{\mathbf{a}_i^\top \mathbf{x}_k} \right) \Big].
$$

- The term $-\mathbf{a}_i^\top \mathbf{x}_{b_i}$ is linear in $\mathbf{X}$, hence convex.  
- The second term is $\phi(\mathbf{u})$ with $\mathbf{u} = [\mathbf{a}_i^\top \mathbf{x}_1, \dots, \mathbf{a}_i^\top \mathbf{x}_C]$, which is convex since $\phi$ is convex and linear maps preserve convexity.  
- A nonnegative sum of convex functions is convex.

Therefore, $f(\mathbf{X})$ is convex. $\blacktriangle$

You have just established that the negative log-likelihood is a convex function. So in principle, any local minimum of the maximum likelihood estimator
$$
\mathbf{X}^\star_{ML} = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X})
$$

is a global minimum. But does the minimum always exist? We will ponder this question in the following three points.


__(c)__ (1 point) Explain the difference between infima and minima. Give an example of a convex function on $\mathbb{R}$ that does not attain its infimum.


**Answer.**
The infimum of a function $f$ over a domain $D$ is the greatest lower bound of the set $\{f(x) : x \in D\}$.  
Formally, $\alpha = \inf_{x \in D} f(x)$ if:
1. $\alpha \leq f(x)$ for all $x \in D$, and  
2. for every $\varepsilon > 0$, there exists $x \in D$ such that $f(x) < \alpha + \varepsilon$.

Intuitively, the first condition ensures that $\alpha$ is indeed less or equal than all values of the function $f$, and the second condition ensures "asymptotical behaviour" of the function $f$ towards the infimum.

Note that infimum always exists.  

The minimum of $f$ over $D$ is a value $\beta$ such that:
1. $\beta = f(x^*)$ for some $x^* \in D$, and  
2. $\beta \leq f(x)$ for all $x \in D$.  

Equivalently, a minimum exists if and only if the infimum is attained at some 
$x^* \in D$, in which case
$$
\min_{x \in D} f(x) = f(x^*) = \inf_{x \in D} f(x).
$$


**Example.**  
Consider the convex function
$$
f(x) = e^x, \quad x \in \mathbb{R}.
$$
We have
$$
\inf_{x \in \mathbb{R}} f(x) = 0,
$$
but there is no $x \in \mathbb{R}$ such that $e^x = 0$.  
Therefore, $f(x)$ has an infimum but no minimum. $\blacktriangle$


__(d)__ (1 point) Assume there exists $\mathbf{X}_0 \in \mathbb{R}^{p\times C}$ such that for all $i$,
$$
\mathbf{a}_i^\top \mathbf{x}_{0, b_i} - \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k} > 0.
$$
This is called one-versus-all complete separation in multiclass settings. Give a geometric interpretation (e.g., for $p=2$) and explain why the name is appropriate.


**Answer.**  

The condition  
$$
\mathbf{a}_i^\top \mathbf{x}_{0,b_i} - \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k} > 0
$$
means that for each sample $\mathbf{a}_i$, the score for its true class $b_i$ is strictly larger than the score for any other class.  

Geometrically, this implies that there exists a set of separating lines (in $\mathbb{R}^2$) such that each class lies entirely within its own region, separated from the others. In other words, every point is projected (via the inner product) most strongly onto the weight vector of its true class.

Note that this condition implies that exactly one hyperplane (line in $\mathbb{R}^2$) should be enough to isolate one class from all of the others, so if we have a situation in $\mathbb{R}^2$ where, for example, there are 5 clusters, with onein the middle, even though the clusters might me linearly separable there wouldn't exist one hyperplane discriminating the central cluster from the others. 

Thus, this situation is called one-versus-all complete separation because for each class $c$, there exists a separating hyperplane (line in 2D) that perfectly distinguishes class $c$ from all the others combined. Each class is linearly separable against the union of the remaining classes.
$\blacktriangle$




From this, you should see that it is likely that some datasets satisfy the complete separation assumption. Unfortunately, as you will show next, this can become an obstacle.


__(e)__ (1 point) In a one-versus-all complete separation setting (as in (d)), prove that $f$ does not attain its minimum. Hint: consider $f(\alpha \mathbf{X}_0)$ as $\alpha \to +\infty$ and compare it to $f(\mathbf{X}_0)$.


**Answer.**  
At the beginning, we will recall the negative log-likelihood
$$
f(\mathbf{X}) 
= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} 
+ \log \Big( \sum_{k=1}^C e^{\mathbf{a}_i^\top \mathbf{x}_k} \Big) \right].
$$

From part (d), we also recall the setting: there exists $\mathbf{X}_0$ such that for every $i$,
$$
\mathbf{a}_i^\top \mathbf{x}_{0,b_i} > \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k}.
$$

---

Firstly, take $\alpha \mathbf{X}_0$ with $\alpha > 0$, with $\mathbf{X}_0$ satisfying the condition. For sample $i$, the term inside the sum becomes
$$
\begin{aligned}
-\alpha \mathbf{a}_i^\top \mathbf{x}_{0,b_i}
+ \log \Big( \sum_{k=1}^C e^{\alpha \mathbf{a}_i^\top \mathbf{x}_{0,k}} \Big).
\end{aligned}
$$

Now, we factor out the largest exponent, corresponding to the true class $b_i$:
$$
\begin{aligned}
&= -\alpha \mathbf{a}_i^\top \mathbf{x}_{0,b_i}
+ \log \Bigg( e^{\alpha \mathbf{a}_i^\top \mathbf{x}_{0,b_i}}
\Big( 1 + \sum_{k \neq b_i} e^{\alpha(\mathbf{a}_i^\top \mathbf{x}_{0,k} - \mathbf{a}_i^\top \mathbf{x}_{0,b_i})} \Big) \Bigg).
\end{aligned}
$$

---

This equals
$$
\log \Big( 1 + \sum_{k \neq b_i} e^{\alpha(\mathbf{a}_i^\top \mathbf{x}_{0,k} - \mathbf{a}_i^\top \mathbf{x}_{0,b_i})} \Big).
$$

Since by assumption $\mathbf{a}_i^\top \mathbf{x}_{0,k} - \mathbf{a}_i^\top \mathbf{x}_{0,b_i} < 0$ for all $k \neq b_i$, each exponential term goes to zero as $\alpha \to +\infty$.  

Thus,
$$
\lim_{\alpha \to +\infty} \Big[ -\alpha \mathbf{a}_i^\top \mathbf{x}_{0,b_i}
+ \log \Big( \sum_{k=1}^C e^{\alpha \mathbf{a}_i^\top \mathbf{x}_{0,k}} \Big) \Big] = 0.
$$

---

Summing over all $i$ (finite number of samples) we get
$$
\lim_{\alpha \to +\infty} f(\alpha \mathbf{X}_0) = 0.
$$

Since $f(\mathbf{X}) \geq 0$ always (because it's log likelihood), the infimum of $f$ is $0$.  
But this value is never attained at any finite $\mathbf{X}$, because each term is strictly positive for finite $\alpha$.  

Therefore, with the complete separation assumption, $f$ does not attain its minimum. $\blacktriangle$


We resolve this issue by adding a regularizer. Consider the regularized function

$$
 f_\mu(\mathbf{X}) = f(\mathbf{X}) + \frac{\mu}{2} \|\mathbf{X}\|_F^2, \quad \mu > 0.
$$

__(f)__ (1 point) Show that the gradient with respect to $\mathbf{X}$ of $f_\mu$ can be expressed as
$$
 \nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \sum_{i=1}^n \big( \mathbf{p}_i - \mathbf{e}_{b_i} \big) \mathbf{a}_i^\top + \mu \mathbf{X},\tag{1}
$$
where $\mathbf{e}_{b_i} \in \mathbb{R}^C$ is the [one-hot vector](https://en.wikipedia.org/wiki/One-hot) for class $b_i$, $\mathbf{p}_i \in \mathbb{R}^C$ has entries $p_{i,c} = \mathbb{P}(b_i=c\mid \mathbf{a}_i)$ under the softmax model, and $(\mathbf{p}_i - \mathbf{e}_{b_i})\mathbf{a}_i^\top$ denotes the outer product.



**Answer.**  

We want to compute the gradient of
$$
f_\mu(\mathbf{X}) 
= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} 
+ \log \Big( \sum_{k=1}^C e^{\mathbf{a}_i^\top \mathbf{x}_k} \Big) \right]
+ \frac{\mu}{2} \|\mathbf{X}\|_F^2.
$$

---

At the beginning, we recall that the Frobenius norm of a matrix $\mathbf{X} \in \mathbb{R}^{n \times p}$ is defined as  
$$
\|\mathbf{X}\|_F 
:= 
\|\mathbf{X}\|_F = \left( \sum_{i=1}^{r} \sigma_i^2 \right)^{1/2}=
\left( \sum_{i=1}^{n} \sum_{j=1}^{p} |x_{ij}|^2 \right)^{1/2}
= \left( \operatorname{tr}(\mathbf{X}^\top \mathbf{X}) \right)^{1/2}.
$$


---

**Step 1. Gradient of the regularizer term**

First, we solve the term 
$$
\frac{\mu}{2}\|\mathbf{X}\|_F^2 
= \frac{\mu}{2}\sum_{i=1}^{n}\sum_{j=1}^{p} x_{ij}^2.
$$

Taking the derivative with respect to each element $x_{ij}$ gives  
$$
\frac{\partial}{\partial x_{ij}} 
\left( \frac{\mu}{2}\sum_{i,j} x_{ij}^2 \right)
= \frac{\mu}{2} \cdot 2x_{ij}
= \mu x_{ij}.
$$

Stacking these elementwise derivatives back into matrix form allows us to write  
$$
\nabla_{\mathbf{X}} \frac{\mu}{2}\|\mathbf{X}\|_F^2 = \mu \mathbf{X}.
$$

Hence, the gradient of the Frobenius norm squared term is proportional to the matrix itself.


---

**Step 2. Gradient of the "main" term.**  
For a fixed sample $i$, we will consider
$$
g_i(\mathbf{X}) = -\mathbf{a}_i^\top \mathbf{x}_{b_i} 
+ \log \Big( \sum_{k=1}^C e^{\mathbf{a}_i^\top \mathbf{x}_k} \Big).
$$

Here, $\mathbf{a}_i \in \mathbb{R}^d$ is the feature vector for sample $i$,  
and $\mathbf{X} = [\mathbf{x}_1, \dots, \mathbf{x}_C] \in \mathbb{R}^{d \times C}$ is the parameter matrix, where each column $\mathbf{x}_c \in \mathbb{R}^d$ corresponds to class $c$.

Since $\mathbf{X}$ is a matrix whose columns are $\mathbf{x}_1, \dots, \mathbf{x}_C$, the gradient with respect to $\mathbf{X}$ can be obtained by stacking the column-wise gradients:
$$
\nabla_{\mathbf{X}} g_i(\mathbf{X}) 
= \big[\, \nabla_{\mathbf{x}_1} g_i, \, \nabla_{\mathbf{x}_2} g_i, \, \dots, \, \nabla_{\mathbf{x}_C} g_i \,\big].
$$

---

#### Derivative of the first term

The gradient of the linear term $-\mathbf{a}_i^\top \mathbf{x}_{b_i}$ with respect to each $\mathbf{x}_c$ is
$$
\nabla_{\mathbf{x}_c}\big(-\mathbf{a}_i^\top \mathbf{x}_{b_i}\big)
= \begin{cases}
- \mathbf{a}_i, & c = b_i, \\
0, & c \neq b_i.
\end{cases}
$$

Since the gradient with respect to the full matrix $\mathbf{X}$ can be written by stacking the column-wise derivatives, this expression can be represented compactly using the one-hot vector $\mathbf{e}_{b_i} \in \mathbb{R}^C$ as
$$
\nabla_{\mathbf{X}}\big(-\mathbf{a}_i^\top \mathbf{x}_{b_i}\big)
= -\mathbf{a}_i \mathbf{e}_{b_i}^\top.
$$

---

#### Derivative of the second term (log-sum-exp)

Define
$$
p_{i,c} = \frac{e^{\mathbf{a}_i^\top \mathbf{x}_c}}
{\sum_{k=1}^C e^{\mathbf{a}_i^\top \mathbf{x}_k}}, 
\quad \mathbf{p}_i = [p_{i,1}, \dots, p_{i,C}]^\top.
$$

Then, by the chain rule,
$$
\nabla_{\mathbf{x}_c} 
\log \Big( \sum_{k=1}^C e^{\mathbf{a}_i^\top \mathbf{x}_k} \Big)
= \frac{1}{\sum_{k=1}^C e^{\mathbf{a}_i^\top \mathbf{x}_k}}
\cdot e^{\mathbf{a}_i^\top \mathbf{x}_c} \, \mathbf{a}_i
= p_{i,c}\,\mathbf{a}_i.
$$

Stacking these for all $c$ gives the gradient with respect to $\mathbf{X}$:
$$
\nabla_{\mathbf{X}} 
\log \Big( \sum_{k=1}^C e^{\mathbf{a}_i^\top \mathbf{x}_k} \Big)
= \mathbf{a}_i \mathbf{p}_i^\top.
$$

---

#### Full contribution from sample $i$

Combining the two parts — the derivative of the linear term $(-\mathbf{a}_i^\top \mathbf{x}_{b_i})$ and the log-sum-exp term gives:
$$
\nabla_{\mathbf{X}} g_i(\mathbf{X})
= \mathbf{a}_i (\mathbf{p}_i - \mathbf{e}_{b_i})^\top.
$$

This represents the contribution to the total gradient from sample $i$.  
Summing over all samples and adding the regularization term from **Step 1** yields the full gradient of the loss function.

---

**Step 3. Combine over all samples.**  
Summing over $i = 1,\dots,n$, we get
$$
\nabla_{\mathbf{X}} f_\mu(\mathbf{X})
= \sum_{i=1}^n \mathbf{a}_i (\mathbf{p}_i - \mathbf{e}_{b_i})^\top + \mu \mathbf{X}.
$$

This (luckily :)) matches the desired expression. $\blacktriangle$

__(g)__ (1 point) Show that the Hessian of $f_\mu$ can be written as
$$
 \nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I},\tag{2}
$$
where $\otimes$ is the Kronecker product, and $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top$ is the softmax Jacobian, which is positive semidefinite.


**Answer.**

The Hessian of  $f_\mu$ is the gradient with respect to $\mathbf{X}$ of $\nabla f_\mu(\mathbf{X})$. 
Therefore we are now trying to find the gradient of the following expression:
$$
\nabla_{\mathbf{X}} f_\mu(\mathbf{X})
= \sum_{i=1}^n (\mathbf{p}_i - \mathbf{e}_{b_i}) \mathbf{a}_i^\top + \mu \mathbf{X}.
$$

Let us decompose it into two terms. The first term being 
$$g(\mathbf{X}) = \sum_{i=1}^n (\mathbf{p}_i - \mathbf{e}_{b_i}) \mathbf{a}_i^\top $$
and the second term being 
$$h(\mathbf{X}) = \mu \mathbf{X}.$$

We will now find the gradient by summing up the gradients of both terms :

$$ \nabla^2_{\mathbf{X}} f_\mu(\mathbf{X}) = \nabla_{\mathbf{X}} g(\mathbf{X}) +  \nabla_{\mathbf{X}} h(\mathbf{X})
$$
As for the $h(\mathbf{X})$ we have
$$\nabla_{\mathbf{X}} h(\mathbf{X}) = \mu \mathbf{I}
$$

In $g(\mathbf{X})$ the only part that is dependent of $\mathbf{X}$ is $\sum_{i=1}^n \mathbf{p}_i(\mathbf{X}) \mathbf{a}_i^\top $ and next we will find a gradient of this term.

First we look into how does $\mathbf{p}_i$ depend on $\mathbf{X}$. 
Let 
$$
z_{i,c}=\mathbf a_i^\top \mathbf x_c ,    
  \mathbf z_i=[z_{i,1},\dots,  z_{i,C}]^\top, 
$$
be a score which is an argument of the exponent in the softmax formula. From there, we can rewrite the softmax as follows:
$$
 \mathbf (p_i)_c = \dfrac{e^{z_{i,c}}}{\sum_{k=1}^C e^{z_{i,k}}}.
$$


Now we calucate the Jacobian of the function $\mathbf (p_i)_c(z)$ with respect to $z$
$$
\frac{\partial (\mathbf p_i)_c}{\partial z_{i,\ell}}
= (\mathbf p_i)_c\big(\delta_{c\ell}-(\mathbf p_i)_\ell\big)
$$
where  
$$
\delta_{c\ell} =
\begin{cases}
\mathbf 1, & c=\ell,\\
\mathbf 0, & c\neq \ell.
\end{cases}
$$

If we make a diagonal matrix $J \in \mathbb{R}^{c\times c} $ and set $(\mathbf p_i)_c$ as diagonal elements and in non-diagonal elements we put $- (\mathbf p_i)_c (\mathbf p_i)_\ell $ we get the following notation. The righthand side represents the softmax Jacobian $J_{\mathbf p_i}(z)$
$$
\frac{\partial (\mathbf p_i)_c}{\partial z_{i,\ell}}
= (\mathbf p_i)_c\big(\delta_{c\ell}-(\mathbf p_i)_\ell\big)
\quad\Longleftrightarrow\quad
\frac{\partial \mathbf p_i}{\partial \mathbf z_i}
=\operatorname{Diag}(\mathbf p_i)-\mathbf p_i\mathbf p_i^\top
=: J_{\mathbf p_i}(z).
$$

Next we caluclate the Jacobian of $z_{i,r}$ with respect to $x_l$

$$
z_{i,r}=\mathbf a_i^\top \mathbf x_r
\ \Rightarrow\
\frac{\partial z_{i,r}}{\partial \mathbf x_\ell}
=
\begin{cases}
\mathbf a_i, & r=\ell,\\
\mathbf 0, & r\neq \ell.
\end{cases}
$$

Therefore in $z_i$, vector $x_l$ affects only $z_{i,l}$.

Now, we apply the chain rule for a fixed component $c$ and fixed $l$:
$$
\frac{\partial (\mathbf p_i)_c}{\partial \mathbf x_\ell}
=\sum_{r=1}^C 
\frac{\partial (\mathbf p_i)_c}{\partial z_{i,r}}
\frac{\partial z_{i,r}}{\partial \mathbf x_\ell}
=\frac{\partial (\mathbf p_i)_c}{\partial z_{i,\ell}}\ \mathbf a_i
=(\mathbf p_i)_c\big(\delta_{c\ell}-(\mathbf p_i)_\ell\big)\ \mathbf a_i.
$$


Recall the gradient of $f(X)$ for each column:

$$
\nabla_{\mathbf{X_c}} f_\mu(\mathbf{X})
= \sum_{i=1}^n ((\mathbf{p}_i)_c - \mathbf{e}_{b_i=c}) \mathbf{a}_i.
$$





Hessian block element $c,l$ is given by

$$
\frac{\partial^2 f}{\partial \mathbf x_c\,\partial \mathbf x_\ell^\top}
=
\sum_{i=1}^n \frac{\partial (\mathbf p_i)_c}{\partial \mathbf x_\ell}\,\mathbf a_i^\top
\;=\;
\sum_{i=1}^n \Big[(\mathbf p_i)_c\big(\delta_{c\ell}-(\mathbf p_i)_\ell\big)\,\mathbf a_i\Big]\mathbf a_i^\top
\;=\;
\sum_{i=1}^n (\mathbf p_i)_c\big(\delta_{c\ell}-(\mathbf p_i)_\ell\big)\,\mathbf a_i\mathbf a_i^\top.
$$

That gives us the block-matrix of $C \times C$  blocks, each of dimension $p \times p$.



$$
\nabla^2 f(\mathbf X)
=\sum_{i=1}^n \big(\mathbf a_i\mathbf a_i^\top\big)\ \otimes\
\Big(\operatorname{Diag}(\mathbf p_i)-\mathbf p_i\mathbf p_i^\top\Big).
$$
As $ f_\mu=f+\tfrac{\mu}{2}\|\mathbf X\|_F^2 $, then
$$
\nabla^2 f_\mu(\mathbf X)=\nabla^2 f(\mathbf X)+\mu\,\mathbf I.
$$

Let $X\in\mathbb{R}^{p\times C}$, samples $a_i\in\mathbb{R}^p $, labels $b_i\in\{1,\dots,C\}$.  
Define logits and probabilities:


__(h)__ (1 point) Show that $f_\mu$ is $\mu$-strongly convex.


**Answer.**

Recall

$$
 f_\mu(\mathbf{X}) = f(\mathbf{X}) + \frac{\mu}{2} \|\mathbf{X}\|_F^2, \quad \mu > 0.
$$

From part (b) we have that $f$ is convex. 

The Hessian $ \nabla^2_{\mathbf{X}} f_\mu(\mathbf{X}) $ satisfies:

$$
\nabla^2_{\mathbf{X}} f_\mu(\mathbf{X}) = \nabla^2_{\mathbf{X}} f(\mathbf{X}) + \mu \mathbf{I}
$$

Since we have that $f$ is convex, $\nabla^2_{\mathbf{X}} f(\mathbf{X}) $ is positive semidefinite, which means that all eigenvalues of $\nabla^2_{\mathbf{X}} f(\mathbf{X}) $ are greater or equal to zero. Then, by adding the Hessian of the regularizer, we assure that no eigenvalue is less than $\mu$. We prove that as follows:


Let $H(X) := \nabla^2 f(X) \in \mathbb{R}^{d\times d}$ be **symmetric** and **PSD** (from convexity of $f$).  

Fix any $X$. Take the eigendecomposition
$$
H(X) \;=\; Q\,\Lambda\,Q^\top,\qquad
Q^\top Q = I,\qquad
\Lambda=\operatorname{Diag}(\lambda_1,\dots,\lambda_d),\quad \lambda_i \ge 0.
$$
 
Consider the ridge-augmented Hessian, which is the Hessian of our funciton $ f_\mu(\mathbf{X})$:
$$
H_\mu(X) \;:=\; \nabla^2 f_\mu(X) \;=\; \nabla^2 f(X) + \mu I \;=\; H(X) + \mu I,\qquad \mu>0.
$$

Because $I = QQ^\top$ commutes with every matrix, we have that:
$$
H_\mu(X) \;=\; Q\,\Lambda\,Q^\top + \mu\, Q I Q^\top 
= Q(\Lambda + \mu I)Q^\top
= Q\,\operatorname{Diag}(\lambda_1+\mu,\dots,\lambda_d+\mu)\,Q^\top .
$$

Hence the eigenvalues are shifted by $\mu$: 
$$\lambda_i(H_\mu(X)) = \lambda_i(H(X)) + \mu \ge \mu.$$
## dODAJ lambai - mu >=0
A twice-differentiable function is $\mu$-strongly convex iff $\nabla^2 f_\mu(X) \succeq \mu I$ for all $X$.  
Since all eigenvalues of $H_\mu(X)$ are $\ge \mu$, we have
$$
\nabla^2 f_\mu(X) \succeq \mu I \quad \text{for all } X,
$$
which is sufficient for the function to be  $\mu$-strongly convex.


__(i)__ (1 point) Is it possible for a strongly convex function to not attain its minimum? Justify your reasoning (you may assume the domain is $\mathbb{R}^{p\times C}$).


Let $f$ be a $\mu$-strongly convex function on $ dom(f) = \mathbb{R}^{p\times C}$. The defintion of strong convexity gives us

$f(y) \geq f(x) + \nabla f(x)^T(y - x) + \frac{\mu}{2} \Vert (y - x) \Vert_2^2; \forall x,y \in dom(f)$.

We also have that:

$\nabla^2f(x) \succcurlyeq  0$

Strong convexity of a fucntion, implies a quadratic lower bound. 



We will now show that $f_\mu$ is smooth, i.e., $\nabla f_\mu$ is L-Lipschitz with respect to the Frobenius norm, with a simple conservative bound
$$
 L = \|\mathbf{A}\|_F^2 + \mu.
$$
where
$$
 \mathbf{A} = \begin{bmatrix}
  \leftarrow &  \mathbf{a}_1^\top & \rightarrow \\
  \leftarrow &  \mathbf{a}_2^\top & \rightarrow \\
   &  \ldots &  \\
  \leftarrow &  \mathbf{a}_n^\top & \rightarrow \\
 \end{bmatrix}.
$$
(You may use that the operator norm of the softmax Jacobian is bounded by 1/4, and a looser bound $\le 1$ is acceptable for grading.)

Hint: check the properties of the spectral norm with respect to dot product, Kronecker product, and outer product.

(1 point for all three questions)


__(j-1)__ Show that $\lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^T) = \left\| \mathbf{a}_i\right\|_2^2$, where $\lambda_{\max}(\cdot)$ denotes the largest eigenvalue.


**Answer.**

Let $A_i = \mathbf{a}_i\mathbf{a}_i^T \in \mathbb{R}^{p\times p}$.

We have that $ \mathbf{a}_i^T\mathbf{a}_i =  \left\| \mathbf{a}_i\right\|_2^2$. Then

$$A_i\mathbf{a}_i = (\mathbf{a}_i\mathbf{a}_i^T)\mathbf{a}_i = \mathbf{a}_i(\mathbf{a}_i^T)\mathbf{a}_i) = \left\| \mathbf{a}_i\right\|_2^2 \mathbf{a}_i$$

From here we can conclude that $\mathbf{a}_i$ is an eigenvector with an eigenvalue equal to $\left\| \mathbf{a}_i\right\|_2^2$. So indeed, $\left\| \mathbf{a}_i\right\|_2^2$ is an eigenvalue of $A_i = \mathbf{a}_i\mathbf{a}_i^T$

We now need to show that it is the largest one.

All the eigenvectors are orthogonal. Therefore, for any other eigenvector $u$ we have $u \perp  \mathbf{a}_i$, that is $ \mathbf{a}_i^T u = 0$ we have

$$
A_i u = (\mathbf{a}_i\mathbf{a}_i^T)u = \mathbf{a}_i(\mathbf{a}_i^Tu) = 0.
$$

This means that for all the other p-1 eigenvectors, the corresponding eigenvalues are 0. Thus, $\left\| \mathbf{a}_i\right\|_2^2$ is the largest eigenvalue of $A_i$, that is of $\mathbf{a}_i\mathbf{a}_i^T$. $\blacktriangle$

__(j-2)__ Using (2), show that $\lambda_{\max}(\nabla^2 f_\mu(\mathbf{X})) \leq \sum_{i=1}^{n} \|\mathbf{a}_i\|_2^2 + \mu$.

**Answer.**

Recall (2):

$$
 \nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I},\tag{2}
$$


__(j-3)__ Conclude that $f_\mu$ is $L$-smooth for $L = \|\mathbf{A}\|_F^2 + \mu$.


__(l)__ (1 point) KL divergence and NLL. Let $q(b_i\mid\mathbf{a}_i)$ be the true label distribution and $p(b_i\mid\mathbf{a}_i)$ the model softmax. Write the KL divergence $\mathrm{KL}(q\,\|\,p)$ and show that minimizing the KL divergence between $q$ and $p$ is equivalent to minimizing the negative log-likelihood derived in (a).


From your work in this section, you have shown that the maximum likelihood estimator for multiclass softmax logistic regression might not exist, but it can be guaranteed to exist by adding a $\|\cdot\|_F^2$ regularizer. Consequently, the estimator for $\mathbf{X}$ we will use will be the solution of the smooth strongly convex problem,
$$
 \mathbf{X}^\star = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X}) + \frac{\mu}{2}\|\mathbf{X}\|_F^2.\tag{3}
$$


## Binary logistic regression (specialization for Part 2)

While this part analyzed the multiclass (softmax) setting, in the next exercise we will continue under the simplified two-class case.

Let labels be $b_i \in \{-1, +1\}$, features $\mathbf{a}_i \in \mathbb{R}^p$, and weight vector $\mathbf{x} \in \mathbb{R}^p$. Define the sigmoid
$$
\sigma(t) = \frac{1}{1+e^{-t}}.
$$
Model the conditional distribution as
$$
\mathbb{P}(b_i = j \mid \mathbf{a}_i) = \sigma\big(j\, \mathbf{a}_i^\top \mathbf{x}\big), \quad j \in \{-1,+1\}.
$$
The likelihood over i.i.d. samples $\{(\mathbf{a}_i, b_i)\}_{i=1}^n$ is
$$
\mathcal{L}(\mathbf{x}) = \prod_{i=1}^n \sigma\big(b_i\, \mathbf{a}_i^\top \mathbf{x}\big),
$$
so the negative log-likelihood is
$$
 f(\mathbf{x}) = -\log \mathcal{L}(\mathbf{x}) = \sum_{i=1}^n \log\big(1 + e^{-b_i\, \mathbf{a}_i^\top \mathbf{x}}\big).
$$

__(m)__ (1 point) Show that the gradient of the negative log-likelihood is the standard binary logistic regression gradient:
$$
\nabla f(\mathbf{x}) = \sum_{i=1}^n \big(-b_i\, \sigma(-b_i\, \mathbf{a}_i^\top \mathbf{x})\big)\, \mathbf{a}_i.
$$
(Hint: use the chain rule and $\sigma'(t) = \sigma(t)\big(1-\sigma(t)\big)$.)

We will use this binary formulation in Part 2 - First order methods.


**Answer.**  
We have
$$
f(\mathbf{x}) =  \sum_{i=1}^n \log\!\big(\sigma(b_i \mathbf{a}_i^\top \mathbf{x})^{-1}\big).
$$

Taking the derivative over coordinate $x_j$, and using the substitution  $t_i = b_i \mathbf{a}_i^\top \mathbf{x}$ for simplicity we have
$$
\frac{\partial}{\partial x_j}\log(\sigma(t_i)^{-1})
= -\frac{\sigma'(t_i)}{\sigma(t_i)} \cdot \frac{\partial t_i}{\partial x_j}.
$$

Using $\sigma'(t) = \sigma(t)(1-\sigma(t))$ and $\tfrac{\partial t_i}{\partial x_j}=b_i a_{ij}$,
$$
\frac{\partial f}{\partial x_j}
= \sum_{i=1}^n -b_i \, (1-\sigma(t_i)) \, a_{ij}.
$$

Since $t_i = b_i \mathbf{a}_i^\top \mathbf{x}$,
$$
\frac{\partial f}{\partial x_j}
= \sum_{i=1}^n \big(-b_i \, \sigma(-b_i \mathbf{a}_i^\top \mathbf{x}) \, a_{ij}\big).
$$

At the end, we obtain a final form (by stacking derivatives over coordinates into a vector form)
$$
\nabla f(\mathbf{x})
    = \sum_{i=1}^n \big(-b_i \, \sigma(-b_i \mathbf{a}_i^\top \mathbf{x}) \, \big) \mathbf{a}_i. \;\blacktriangle
$$
